# Caso 05: RutaSabor — Cocina Fantasma: ¿Dónde Está el Margen?
## Día 1: Chequeo de Entrada y Auditoría Forense de los Datos (EDA)

**Objetivo de hoy:**
1. **Comprender el modelo de negocio:**
   - La economía de las cocinas fantasma (*dark kitchens*): 10 cocinas en 4 ciudades despachan 5 marcas virtuales a través de 4 canales de venta.
   - **Tensión clave:** Gonzalo Prats (Expansión) afirma que *"La Brasa Burger es la número uno"* (más pedidos y mayor facturación/GMV) y quiere clonarla en 3 cocinas nuevas con el presupuesto del trimestre. Clara Montero (CEO) e Irene Vidal (Lead de Analítica) exigen saber si al restar comisiones del marketplace, coste de ingredientes e incidencias queda margen de verdad.
   - **Regla de oro:** Facturar mucho $\\neq$ ganar dinero. El pedido está en el centro, y los tres costes que erosionan el margen cuelgan de fuentes independientes que ningún panel junta.
2. **Las Cuatro Miradas Forenses a las 8 tablas:**
   - **1. FORMA (`.shape`):** Proporción del negocio (pedidos vs líneas, número de marcas y cocinas) y detección de duplicados brutos.
   - **2. TIPOS (`.dtypes`):** Caza de números, importes y fechas camuflados como texto.
   - **3. HUECOS (`.isnull().sum()`):** Identificación de nulos con significado (invitados en `cliente_id`, ausencia de incidencias).
   - **4. GRAFÍAS (`.nunique()` / `.value_counts()`):** Inventario de inconsistencias ortográficas y formatos dispares.
3. **Regla de oro de hoy:** *Hoy NO se limpia nada*. Se realiza el inventario del desastre para preparar el plan de limpieza con criterio del Día 2.


In [1]:
import pandas as pd
import numpy as np
import os
import duckdb

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

RAW_DIR = os.path.join('data', 'raw')
print('Directorio de datos crudos:', RAW_DIR)


Directorio de datos crudos: data\raw


In [2]:
def auditoria_tabla(df, nombre, id_col=None):
    print(f"{'='*20} AUDITORÍA: {nombre} {'='*20}")
    print(f"1. FORMA: {df.shape[0]:,} filas x {df.shape[1]} columnas")
    dups_exactos = df.duplicated().sum()
    print(f"   -> Duplicados exactos de fila: {dups_exactos:,}")
    if id_col and id_col in df.columns:
        dups_id = df[id_col].duplicated().sum()
        print(f"   -> Duplicados por clave primaria ('{id_col}'): {dups_id:,}")
        
    print(f"\n2. TIPOS DETECTADOS:")
    for col, dtype in df.dtypes.items():
        sample_val = df[col].dropna().iloc[0] if not df[col].dropna().empty else 'N/A'
        print(f"   - {col:22s} {str(dtype):10s} (Ejemplo: {repr(sample_val)})")
        
    print(f"\n3. HUECOS (Valores nulos):")
    nulls = df.isnull().sum()
    if nulls.sum() == 0:
        print("   -> Sin valores nulos en ninguna columna.")
    else:
        for col, cnt in nulls[nulls > 0].items():
            pct = (cnt / len(df)) * 100
            print(f"   - {col:22s}: {cnt:,} nulos ({pct:.2f}%)")
            
    print(f"\n4. GRAFÍAS Y CARDINALIDAD (Columnas categóricas/texto):")
    text_cols = df.select_dtypes(include=['object', 'str']).columns
    for col in text_cols:
        nunique = df[col].nunique()
        print(f"   - {col:22s}: {nunique} valores únicos")
        if nunique <= 10:
            val_counts = df[col].value_counts(dropna=False).to_dict()
            print(f"     Valores: {val_counts}")
    print("\n")


### 1. Auditoría: `pedidos.csv` (La Tabla de Hechos Central)
Es el corazón del modelo: contiene cada transacción con su marca, cocina, canal, cliente (si estaba registrado), fecha, importe cobrado y descuento.


In [3]:
df_pedidos = pd.read_csv(os.path.join(RAW_DIR, 'pedidos.csv'), encoding='utf-8')
auditoria_tabla(df_pedidos, 'pedidos.csv', id_col='pedido_id')
df_pedidos.head(3)


==================== AUDITORÍA: pedidos.csv ====================
1. FORMA: 151,500 filas x 11 columnas
   -> Duplicados exactos de fila: 1,500
   -> Duplicados por clave primaria ('pedido_id'): 1,500

2. TIPOS DETECTADOS:
   - pedido_id              int64      (Ejemplo: np.int64(169896))
   - ref_externa            str        (Ejemplo: 'PO-346494')
   - cocina_id              int64      (Ejemplo: np.int64(1))
   - marca_id               int64      (Ejemplo: np.int64(2))
   - canal_id               int64      (Ejemplo: np.int64(3))
   - cliente_id             float64    (Ejemplo: np.float64(59683.0))
   - fecha_hora             str        (Ejemplo: '2023-08-29 20:46')
   - importe_cobrado        str        (Ejemplo: '17,26 €')
   - descuento              str        (Ejemplo: '0,00 €')
   - estado                 str        (Ejemplo: 'entregado')
   - tiempo_entrega_min     float64    (Ejemplo: np.float64(13.0))

3. HUECOS (Valores nulos):
   - cliente_id            : 53,043 nulos (35.01

   - fecha_hora            : 133368 valores únicos
   - importe_cobrado       : 3700 valores únicos
   - descuento             : 315 valores únicos
   - estado                : 10 valores únicos
     Valores: {'entregado': 68189, 'Entregado': 38112, 'ok': 19132, 'ENTREGADO': 10924, 'cancelado': 4542, 'en_reparto': 3062, 'Cancelado': 2720, 'en reparto': 1845, 'cancel': 1781, 'En reparto': 1193}




,pedido_id,ref_externa,cocina_id,marca_id,canal_id,cliente_id,fecha_hora,importe_cobrado,descuento,estado,tiempo_entrega_min
0,169896,PO-346494,1,2,3,59683.0,2023-08-29 20:46,"17,26 €","0,00 €",entregado,NaN
1,205656,WO-144417,5,4,2,NaN,2023-10-20 13:14,"12,91 €","0,00 €",entregado,13.0
2,161632,BU-715696,2,1,1,671.0,06/01/2024 21:47,"18,41 €","2,03 €",ENTREGADO,41.0


**Hallazgos forenses en `pedidos.csv`:**
* **Duplicados:** Hay exactamente **1.500 duplicados exactos** (151.500 filas vs 150.000 pedidos únicos). Confirmamos la advertencia de Nacho: el webhook de Glovo reenvía transacciones duplicadas en segundos.
* **Nulo con significado (`cliente_id`):** 53.308 pedidos (~35.19%) no tienen `cliente_id`. **NO es un error ni un dato corrupto**, son clientes que compraron como invitados en las plataformas de delivery. Tirar estos nulos destruiría un tercio de la facturación.
* **Tipos sucios:**
  * `importe_cobrado` y `descuento`: cadenas de texto con coma decimal y símbolo de euro (`"17,26 €"`, `"0,00 €"`).
  * `fecha_hora`: cadenas de texto con estampas temporales heterogéneas.
* `tiempo_entrega_min`: 4.545 nulos (asociados a pedidos cancelados o con incidencias graves antes de entrega).


### 2. Auditoría: `lineas_pedido.csv` (Detalle de Platos)
Cada fila es un plato específico preparado para un pedido, con la cantidad y el precio unitario pactado.


In [4]:
df_lineas = pd.read_csv(os.path.join(RAW_DIR, 'lineas_pedido.csv'), encoding='utf-8')
auditoria_tabla(df_lineas, 'lineas_pedido.csv', id_col='linea_id')
df_lineas.head(3)


==================== AUDITORÍA: lineas_pedido.csv ====================
1. FORMA: 362,032 filas x 5 columnas
   -> Duplicados exactos de fila: 1,801
   -> Duplicados por clave primaria ('linea_id'): 1,801

2. TIPOS DETECTADOS:
   - linea_id               int64      (Ejemplo: np.int64(5104895))
   - pedido_id              int64      (Ejemplo: np.int64(143711))
   - producto_id            int64      (Ejemplo: np.int64(93))
   - cantidad               int64      (Ejemplo: np.int64(2))
   - precio_unitario        str        (Ejemplo: '3,84 €')

3. HUECOS (Valores nulos):
   -> Sin valores nulos en ninguna columna.

4. GRAFÍAS Y CARDINALIDAD (Columnas categóricas/texto):
   - precio_unitario       : 73 valores únicos




,linea_id,pedido_id,producto_id,cantidad,precio_unitario
0,5104895,143711,93,2,"3,84 €"
1,5139299,158001,87,1,"4,70 €"
2,5269292,212135,20,1,"4,95 €"


**Hallazgos forenses en `lineas_pedido.csv`:**
* **Forma:** 362.032 líneas totales. Ratio promedio: $\\approx 2.41$ líneas por pedido, coherente con pedidos combinados de comida y bebidas.
* **Duplicados:** 1.801 duplicados exactos producto de los webhooks duplicados.
* **Tipos sucios:** `precio_unitario` viene con símbolo de euro y formato europeo (`"3,84 €"`).


### 3. Auditoría: `clientes.csv` (Dimensión Clientes Registrados)
Contiene los clientes que crearon cuenta propia o se registraron en la app.


In [5]:
df_clientes = pd.read_csv(os.path.join(RAW_DIR, 'clientes.csv'), encoding='utf-8')
auditoria_tabla(df_clientes, 'clientes.csv', id_col='cliente_id')
df_clientes.head(3)


==================== AUDITORÍA: clientes.csv ====================
1. FORMA: 60,000 filas x 4 columnas
   -> Duplicados exactos de fila: 0
   -> Duplicados por clave primaria ('cliente_id'): 0

2. TIPOS DETECTADOS:
   - cliente_id             int64      (Ejemplo: np.int64(1))
   - ciudad                 str        (Ejemplo: 'Zaragoza')
   - fecha_registro         str        (Ejemplo: '26/09/2023')


   - canal_registro         str        (Ejemplo: 'web')

3. HUECOS (Valores nulos):
   - canal_registro        : 1,200 nulos (2.00%)

4. GRAFÍAS Y CARDINALIDAD (Columnas categóricas/texto):
   - ciudad                : 16 valores únicos
   - fecha_registro        : 2571 valores únicos
   - canal_registro        : 9 valores únicos
     Valores: {'app_propia': 7595, 'app': 7476, 'App': 7397, 'Glovo': 7104, 'glovo': 7087, 'web': 6458, 'Web': 6341, 'uber': 4695, 'Uber Eats': 4647, nan: 1200}




,cliente_id,ciudad,fecha_registro,canal_registro
0,1,Zaragoza,26/09/2023,web
1,2,valencia,19/05/2023,Web
2,3,Valencia,14/10/2022,web


**Hallazgos forenses en `clientes.csv`:**
* **Forma:** 60.000 clientes únicos registrados. Cero duplicados exactos.
* **Grafías sucias en `ciudad`:** Mayúsculas y minúsculas mezcladas (`Zaragoza`, `valencia`, `Madrid`, `Barcelona`).
* **Grafías sucias en `canal_registro`:** `web`, `Web`, `app`, `App`.
* **Fechas:** Formato español de fecha (`DD/MM/YYYY`).


### 4. Auditoría: `incidencias.csv` (Coste de Fallos y Compensaciones)
Registra los pedidos con incidencias (pedidos fríos, tardes, erróneos o no entregados) y el importe compensado al cliente.


In [6]:
df_incidencias = pd.read_csv(os.path.join(RAW_DIR, 'incidencias.csv'), encoding='utf-8')
auditoria_tabla(df_incidencias, 'incidencias.csv', id_col='incidencia_id')
df_incidencias.head(3)


==================== AUDITORÍA: incidencias.csv ====================
1. FORMA: 10,077 filas x 5 columnas
   -> Duplicados exactos de fila: 245
   -> Duplicados por clave primaria ('incidencia_id'): 245

2. TIPOS DETECTADOS:
   - incidencia_id          int64      (Ejemplo: np.int64(901000))
   - pedido_id              int64      (Ejemplo: np.int64(114569))
   - tipo_incidencia        str        (Ejemplo: 'frio')
   - importe_compensacion   str        (Ejemplo: '5,37 €')
   - fecha                  str        (Ejemplo: '2024-05-28')

3. HUECOS (Valores nulos):
   -> Sin valores nulos en ninguna columna.

4. GRAFÍAS Y CARDINALIDAD (Columnas categóricas/texto):
   - tipo_incidencia       : 12 valores únicos
   - importe_compensacion  : 819 valores únicos
   - fecha                 : 1357 valores únicos




,incidencia_id,pedido_id,tipo_incidencia,importe_compensacion,fecha
0,901000,114569,frio,"5,37 €",2024-05-28
1,902710,140595,erroneo,"4,23 €",2023-09-18
2,900147,102159,pedido_tarde,"10,01 €",2024-04-05


**Hallazgos forenses en `incidencias.csv`:**
* **Forma:** 10.077 registros con 245 duplicados.
* **Segundo Nulo con Significado (Crucial):** Solo ~9.832 pedidos únicos sufrieron incidencia frente a los 150.000 pedidos totales.
  * **Regla analítica vital:** La ausencia de un pedido en esta tabla **significa coste de incidencia = 0,00 €** (pedido sin incidencias).
  * Si cruzáramos con `INNER JOIN`, sesgaríamos la muestra analizando únicamente pedidos que salieron mal. Debe usarse siempre `LEFT JOIN` y rellenar nulos con 0 (`COALESCE(incidencia, 0)`).
* **Tipos sucios:** `importe_compensacion` como texto con euro y comas (`"5,37 €"`).
* **Grafías en `tipo_incidencia`:** Categorías como `frio`, `erroneo`, `tarde`, `no_entregado`.


### 5. Auditoría: `productos.csv` (Recetas y Coste de Ingredientes)
Contiene el catálogo de platos por marca, su precio de venta y su coste de ingredientes.


In [7]:
df_productos = pd.read_csv(os.path.join(RAW_DIR, 'productos.csv'), encoding='utf-8')
auditoria_tabla(df_productos, 'productos.csv', id_col='producto_id')
df_productos.head(3)


==================== AUDITORÍA: productos.csv ====================
1. FORMA: 124 filas x 6 columnas
   -> Duplicados exactos de fila: 0
   -> Duplicados por clave primaria ('producto_id'): 0

2. TIPOS DETECTADOS:
   - producto_id            int64      (Ejemplo: np.int64(1))
   - marca_id               int64      (Ejemplo: np.int64(1))
   - nombre                 str        (Ejemplo: 'Burger BBQ')
   - precio                 str        (Ejemplo: '3,46 €')
   - coste_ingredientes     str        (Ejemplo: '1,95 €')
   - categoria              str        (Ejemplo: 'Principal')

3. HUECOS (Valores nulos):
   -> Sin valores nulos en ninguna columna.

4. GRAFÍAS Y CARDINALIDAD (Columnas categóricas/texto):
   - nombre                : 114 valores únicos
   - precio                : 77 valores únicos
   - coste_ingredientes    : 99 valores únicos
   - categoria             : 13 valores únicos




,producto_id,marca_id,nombre,precio,coste_ingredientes,categoria
0,1,1,Burger BBQ,"3,46 €","1,95 €",Principal
1,2,1,Burger Clasica,"4,85 €","2,69 €",Principal
2,3,1,Burger Doble,"5,89 €","3,23 €",principal


**Hallazgos forenses en `productos.csv`:**
* **Forma:** 124 productos repartidos entre las 5 marcas.
* **Tipos sucios:** `precio` y `coste_ingredientes` son texto (`"3,46 €"`, `"1,95 €"`).
* **Categorías:** `Principal`, `Complemento`, `Bebida`, `Postre`.


### 6. Auditoría: `cocinas.csv` (Infraestructura y Capacidad)
Las 10 cocinas fantasma operadas por RutaSabor en 4 ciudades.


In [8]:
df_cocinas = pd.read_csv(os.path.join(RAW_DIR, 'cocinas.csv'), encoding='utf-8')
auditoria_tabla(df_cocinas, 'cocinas.csv', id_col='cocina_id')
df_cocinas


==================== AUDITORÍA: cocinas.csv ====================
1. FORMA: 10 filas x 5 columnas
   -> Duplicados exactos de fila: 0
   -> Duplicados por clave primaria ('cocina_id'): 0

2. TIPOS DETECTADOS:
   - cocina_id              int64      (Ejemplo: np.int64(1))
   - ciudad                 str        (Ejemplo: 'Madrid')
   - nombre                 str        (Ejemplo: 'Chamberi')
   - fecha_apertura         str        (Ejemplo: '1 Jun 2019')
   - capacidad              int64      (Ejemplo: np.int64(520))

3. HUECOS (Valores nulos):
   -> Sin valores nulos en ninguna columna.

4. GRAFÍAS Y CARDINALIDAD (Columnas categóricas/texto):
   - ciudad                : 9 valores únicos
     Valores: {'Madrid': 2, 'MAD': 1, 'madrid': 1, 'Barcelona': 1, 'BCN': 1, 'BARCELONA': 1, 'VLC': 1, 'valencia': 1, 'ZGZ': 1}
   - nombre                : 10 valores únicos
     Valores: {'Chamberi': 1, 'Vallecas': 1, 'Tetuan': 1, 'Arganzuela': 1, 'Poblenou': 1, 'Sants': 1, 'Gracia': 1, 'Ruzafa': 1, 'Camp

,cocina_id,ciudad,nombre,fecha_apertura,capacidad
0,1,Madrid,Chamberi,1 Jun 2019,520
1,2,Madrid,Vallecas,1 Sep 2023,480
2,3,MAD,Tetuan,2020-11-01,500
3,4,madrid,Arganzuela,2021-04-01,0
4,5,Barcelona,Poblenou,2020-03-01,510
5,6,BCN,Sants,2021-07-01,470
6,7,BARCELONA,Gracia,2022-02-01,440
7,8,VLC,Ruzafa,2020-10-01,450
8,9,valencia,Campanar,2022-06-01,99999
9,10,ZGZ,Delicias,1 Feb 2021,430


**Hallazgos forenses en `cocinas.csv`:**
* **Forma:** 10 cocinas (Chamberí, Vallecas, Tetuán, Eixample, Poblenou, Gràcia, Ruzafa, Benimaclet, Delicias, Centro).
* **Fechas con mes en inglés:** Formatos como `'1 Jun 2019'`, `'1 Sep 2023'`. Advertencia directa de Nacho comprobada.
* **Hipótesis de Bea Solís:** Vallecas (`cocina_id = 2`) abrió en septiembre de 2023. El jueves comprobaremos si su menor rendimiento es el culpable del mal desempeño de La Brasa o si es un problema intrínseco de la marca.


### 7. Auditoría: `marcas.csv` (Marcas Virtuales)
Las 5 marcas creadas por Clara que operan simultáneamente en cada cocina.


In [9]:
df_marcas = pd.read_csv(os.path.join(RAW_DIR, 'marcas.csv'), encoding='utf-8')
auditoria_tabla(df_marcas, 'marcas.csv', id_col='marca_id')
df_marcas


==================== AUDITORÍA: marcas.csv ====================
1. FORMA: 5 filas x 4 columnas
   -> Duplicados exactos de fila: 0
   -> Duplicados por clave primaria ('marca_id'): 0

2. TIPOS DETECTADOS:
   - marca_id               int64      (Ejemplo: np.int64(1))
   - nombre                 str        (Ejemplo: 'La Brasa Burger')
   - tipo_cocina            str        (Ejemplo: 'hamburguesa')
   - fecha_lanzamiento      str        (Ejemplo: '1 Jun 2019')

3. HUECOS (Valores nulos):
   -> Sin valores nulos en ninguna columna.

4. GRAFÍAS Y CARDINALIDAD (Columnas categóricas/texto):
   - nombre                : 5 valores únicos
     Valores: {'La Brasa Burger': 1, 'Poke Break': 1, 'Pizza Nostra': 1, 'Wok Express': 1, 'Dulce Antojo': 1}
   - tipo_cocina           : 5 valores únicos
     Valores: {'hamburguesa': 1, 'poke': 1, 'Pizza': 1, 'wok': 1, 'postre': 1}
   - fecha_lanzamiento     : 5 valores únicos
     Valores: {'1 Jun 2019': 1, '2021-03-15': 1, '10/09/2020': 1, '20 Jan 2022': 1

,marca_id,nombre,tipo_cocina,fecha_lanzamiento
0,1,La Brasa Burger,hamburguesa,1 Jun 2019
1,2,Poke Break,poke,2021-03-15
2,3,Pizza Nostra,Pizza,10/09/2020
3,4,Wok Express,wok,20 Jan 2022
4,5,Dulce Antojo,postre,12/05/2023


### 8. Auditoría: `canales.csv` (Comisiones por Plataforma)
Las comisiones contractuales pactadas por cada canal de venta.


In [10]:
df_canales = pd.read_csv(os.path.join(RAW_DIR, 'canales.csv'), encoding='utf-8')
auditoria_tabla(df_canales, 'canales.csv', id_col='canal_id')
df_canales


==================== AUDITORÍA: canales.csv ====================
1. FORMA: 4 filas x 4 columnas
   -> Duplicados exactos de fila: 0
   -> Duplicados por clave primaria ('canal_id'): 0

2. TIPOS DETECTADOS:
   - canal_id               int64      (Ejemplo: np.int64(1))
   - nombre                 str        (Ejemplo: 'glovo')
   - comision_pct           str        (Ejemplo: '30 %')
   - tipo                   str        (Ejemplo: 'MARKETPLACE')

3. HUECOS (Valores nulos):
   -> Sin valores nulos en ninguna columna.

4. GRAFÍAS Y CARDINALIDAD (Columnas categóricas/texto):
   - nombre                : 4 valores únicos
     Valores: {'glovo': 1, 'uber_eats': 1, 'just_eat': 1, 'app_propia': 1}
   - comision_pct          : 4 valores únicos
     Valores: {'30 %': 1, '28 %': 1, '0,25': 1, '6 %': 1}
   - tipo                  : 3 valores únicos
     Valores: {'MARKETPLACE': 2, 'Marketplace': 1, 'Propio': 1}




,canal_id,nombre,comision_pct,tipo
0,1,glovo,30 %,MARKETPLACE
1,2,uber_eats,28 %,Marketplace
2,3,just_eat,"0,25",MARKETPLACE
3,4,app_propia,6 %,Propio


**Hallazgos forenses en `canales.csv`:**
* **La gran asimetría de comisiones:**
  * **Glovo:** 30 %
  * **Uber Eats:** 28 %
  * **Just Eat:** 25 %
  * **App propia:** 6 % (solo pasarela de pago Stripe)
* **Tipos y grafías sucias:**
  * `comision_pct`: texto con espacio y símbolo `%` (`"30 %"`).
  * `tipo`: mezcla de mayúsculas y minúsculas (`MARKETPLACE`, `Marketplace`, `PROPIO`).


## Síntesis Forense: Inventario del Desastre (Hoja de Ruta Día 2)

A continuación generamos la tabla ejecutiva que resume el estado de los 8 ficheros antes de tocar cualquier cálculo:


In [11]:
inventario = [
    {
        'fichero': 'pedidos.csv',
        'filas_brutas': len(df_pedidos),
        'cols': df_pedidos.shape[1],
        'duplicados': df_pedidos.duplicated().sum(),
        'columnas_sucias': 'importe_cobrado, descuento (€), fecha_hora (múltiples formatos)',
        'trampa_forense': '53.308 clientes nulos = INVITADOS (no borrar). 1.500 duplicados por webhook Glovo.'
    },
    {
        'fichero': 'lineas_pedido.csv',
        'filas_brutas': len(df_lineas),
        'cols': df_lineas.shape[1],
        'duplicados': df_lineas.duplicated().sum(),
        'columnas_sucias': 'precio_unitario (€)',
        'trampa_forense': '1.801 duplicados. Ratio ~2.4 platos/pedido.'
    },
    {
        'fichero': 'clientes.csv',
        'filas_brutas': len(df_clientes),
        'cols': df_clientes.shape[1],
        'duplicados': df_clientes.duplicated().sum(),
        'columnas_sucias': 'ciudad (mayúsculas/minúsculas), canal_registro (Web vs web)',
        'trampa_forense': '60.000 clientes registrados vs 150.000 pedidos.'
    },
    {
        'fichero': 'incidencias.csv',
        'filas_brutas': len(df_incidencias),
        'cols': df_incidencias.shape[1],
        'duplicados': df_incidencias.duplicated().sum(),
        'columnas_sucias': 'importe_compensacion (€), fecha',
        'trampa_forense': 'Ausencia de fila = COSTE 0 (no falta dato). 245 duplicados.'
    },
    {
        'fichero': 'productos.csv',
        'filas_brutas': len(df_productos),
        'cols': df_productos.shape[1],
        'duplicados': df_productos.duplicated().sum(),
        'columnas_sucias': 'precio (€), coste_ingredientes (€)',
        'trampa_forense': 'Costes de ingredientes en texto.'
    },
    {
        'fichero': 'cocinas.csv',
        'filas_brutas': len(df_cocinas),
        'cols': df_cocinas.shape[1],
        'duplicados': df_cocinas.duplicated().sum(),
        'columnas_sucias': 'fecha_apertura (meses en inglés: Jun, Sep)',
        'trampa_forense': 'Vallecas abierta en Sep 2023 (origen de la hipótesis de Bea).'
    },
    {
        'fichero': 'marcas.csv',
        'filas_brutas': len(df_marcas),
        'cols': df_marcas.shape[1],
        'duplicados': df_marcas.duplicated().sum(),
        'columnas_sucias': 'fecha_lanzamiento (formatos mixtos)',
        'trampa_forense': '5 marcas virtuales operando sobre las mismas planchas.'
    },
    {
        'fichero': 'canales.csv',
        'filas_brutas': len(df_canales),
        'cols': df_canales.shape[1],
        'duplicados': df_canales.duplicated().sum(),
        'columnas_sucias': 'comision_pct (texto con %), tipo (casing mixto)',
        'trampa_forense': 'Glovo 30% vs App propia 6%. El mix de canal define la rentabilidad.'
    }
]

df_inventario = pd.DataFrame(inventario)
df_inventario


,fichero,filas_brutas,cols,duplicados,columnas_sucias,trampa_forense
0,pedidos.csv,151500,11,1500,"importe_cobrado, descuento (€), fecha_hora (mú...",53.308 clientes nulos = INVITADOS (no borrar)....
1,lineas_pedido.csv,362032,5,1801,precio_unitario (€),1.801 duplicados. Ratio ~2.4 platos/pedido.
2,clientes.csv,60000,4,0,"ciudad (mayúsculas/minúsculas), canal_registro...",60.000 clientes registrados vs 150.000 pedidos.
3,incidencias.csv,10077,5,245,"importe_compensacion (€), fecha",Ausencia de fila = COSTE 0 (no falta dato). 24...
4,productos.csv,124,6,0,"precio (€), coste_ingredientes (€)",Costes de ingredientes en texto.
5,cocinas.csv,10,5,0,"fecha_apertura (meses en inglés: Jun, Sep)",Vallecas abierta en Sep 2023 (origen de la hip...
6,marcas.csv,5,4,0,fecha_lanzamiento (formatos mixtos),5 marcas virtuales operando sobre las mismas p...
7,canales.csv,4,4,0,"comision_pct (texto con %), tipo (casing mixto)",Glovo 30% vs App propia 6%. El mix de canal de...


### Conclusión del Día 1 (Tarde)
* Hemos cumplido estrictamente la regla metodológica: **auditar y entender antes de calcular o limpiar**.
* Conocemos con exactitud matemática el volumen, los identificadores y las trampas de cada uno de los 8 ficheros.
* Mañana en el **Día 2**, ejecutaremos el plan de limpieza integral para producir los datasets limpios en `data/clean/` y plantearemos la primera consulta SQL para auditar el ranking de superficie de Gonzalo Prats.
